<a href="https://colab.research.google.com/github/sn0wsally/self-study-Spatial-Transcriptomics/blob/main/Full_pipeline_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Histology-based Breast Tumor Microenvironment Prediction using Spatial Transcriptomics Supervision


---



In [ ]:
from IPython.display import clear_output
clear_output()

### 1. Project Setup

##### 1.1 Install Required Packages

In [ ]:
!pip install -U pandas scanpy squidpy anndata dask leidenalg
!pip install -U torch torchvision tqdm
!pip install -U scikit-learn seaborn pillow

##### 1.2 Imports

In [ ]:
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import scanpy as sc
import squidpy as sq

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import confusion_matrix
import seaborn as sns

warnings.filterwarnings("ignore")

##### 1.3 Reproducibility (Seed Fixing)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

##### 1.4 Device Setup

In [ ]:
def get_device():
    return torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

##### 1.5 Project Folder Creation

In [ ]:
def create_project_dirs():
    base_dir = Path("/content/spatial_tme_project")

    figure_dir = base_dir / "figures"
    patch_dir = base_dir / "patches"
    checkpoint_dir = base_dir / "checkpoints"
    metadata_dir = base_dir / "metadata"

    save_dirs = [
        figure_dir,
        patch_dir,
        checkpoint_dir,
        metadata_dir
    ]

    for path in save_dirs:
        path.mkdir(parents=True, exist_ok=True)

    return {
        "base_dir": base_dir,
        "figure_dir": figure_dir,
        "patch_dir": patch_dir,
        "checkpoint_dir": checkpoint_dir,
        "metadata_dir": metadata_dir
    }

##### 1.6 Global Configuration

In [ ]:
def get_config():
    config = {
        # Dataset
        "sample_id": "V1_Breast_Cancer_Block_A_Section_1",

        # # For unseen data inference
        # "sample_id": "V1_Breast_Cancer_Block_A_Section_2",

        # Patch extraction
        "patch_size": 224,

        # Training
        "batch_size": 16,
        "epochs": 30,
        "learning_rate": 3e-5,
        "weight_decay": 1e-5,
        "patience": 5,  # Early stopping

        # Clustering
        "leiden_resolution": 0.5,
        "n_top_genes": 2000,

        # Final labels
        "num_classes": 3,
        "class_names": [
            "Tumor",
            "Immune",
            "Stroma"
        ],

        # Reproducibility
        "random_seed": 42
    }

    return config

##### 1.7 Summary Printer

In [ ]:
def print_summary(config, device, paths):
    print("\n========== PROJECT CONFIG ==========")
    print(f"Device              : {device}")
    print(f"Sample ID           : {config['sample_id']}")
    print(f"Patch Size          : {config['patch_size']}")
    print(f"Batch Size          : {config['batch_size']}")
    print(f"Epochs              : {config['epochs']}")
    print(f"Learning Rate       : {config['learning_rate']}")
    print(f"Leiden Resolution   : {config['leiden_resolution']}")
    print(f"Classes             : {config['class_names']}")
    print("====================================")

    print("\n========== SAVE PATHS ==========")
    for name, path in paths.items():
        print(f"{name}: {path}")
    print("================================")

##### 1.8 Main Setup Function

In [ ]:
def setup_project():
    config = get_config()

    set_seed(config["random_seed"])

    device = get_device()

    paths = create_project_dirs()

    print_summary(config, device, paths)

    return config, device, paths

##### 1.9 Run

In [ ]:
config, device, paths = setup_project()

### 2. Load Spatial Transcriptomics Data

This section loads the 10x Genomics Visium breast cancer dataset.

This dataset contains:

* H&E tissue image
* Visium spot coordinates
* gene expression matrix
* metadata for spatial analysis

##### 2.1 Load 10x Genomics Visium Dataset

In [ ]:
def load_visium_data(sample_id):
    """
    Load 10x Genomics Visium breast cancer dataset

    Returns
    -------
    adata : AnnData
        Contains:
        - gene expression matrix
        - spatial coordinates
        - H&E image
        - metadata
    """

    print("\nLoading Visium Spatial Transcriptomics dataset...")

    adata = sc.datasets.visium_sge(
        sample_id=sample_id
    )

    # Avoid duplicated gene names
    adata.var_names_make_unique()

    print("Dataset loaded successfully.")
    print(adata)

    return adata

##### 2.2 Dataset Structure Overview

In [ ]:
def print_dataset_info(adata):
    """
    Print important dataset structure info
    """

    print("\n========== DATASET INFO ==========")
    print(f"Number of spots      : {adata.n_obs}")
    print(f"Number of genes      : {adata.n_vars}")

    print("\nobs columns:")
    print(list(adata.obs.columns))

    print("\nobsm keys:")
    print(list(adata.obsm.keys()))

    print("\nuns keys:")
    print(list(adata.uns.keys()))

    print("==================================")

##### 2.3 Basic Sanity Check

In [ ]:
def check_spatial_coordinates(adata):
    """
    Confirm spatial coordinates exist
    """

    if "spatial" not in adata.obsm:
        raise ValueError(
            "Spatial coordinates not found in adata.obsm"
        )

    print("\nSpatial coordinates found:")
    print(adata.obsm["spatial"][:5])

##### 2.4 Visualize Original Tissue + Spot Distribution

In [ ]:
def show_original_tissue(adata):
    """
    Visualize original tissue image + Visium spots
    """

    print("\nPlotting original tissue image...")

    sc.pl.spatial(
        adata,
        color=None,
        title="Original Tissue + Visium Spots",
        alpha_img=0.7
    )

##### 2.5 Quick Marker Gene Visualization

In [ ]:
def show_example_marker_genes(adata):
    """
    Quick biological sanity check
    using known marker genes
    """

    marker_genes = [
        "EPCAM",   # tumor / epithelial
        "CD3D",    # T cells
        "COL1A1"   # stroma / fibroblast
    ]

    existing_markers = [
        gene for gene in marker_genes
        if gene in adata.var_names
    ]

    if len(existing_markers) == 0:
        print("\nNo example marker genes found.")
        return

    print("\nPlotting example marker genes...")

    sc.pl.spatial(
        adata,
        color=existing_markers,
        ncols=len(existing_markers)
    )

##### 2.6 Main Loading Function

In [ ]:
def load_data(config):
    adata = load_visium_data(
        sample_id=config["sample_id"]
    )

    print_dataset_info(adata)

    check_spatial_coordinates(adata)

    show_original_tissue(adata)

    show_example_marker_genes(adata)

    return adata

##### 2.7 Run

In [ ]:
adata = load_data(config)

### 3. Preprocessing and Quality Control

##### 3.1 Mitochondrial Gene Detection

High mitochondrial gene percentage often means:
* damaged spots
* dying cells
* low-quality tissue regions

In [ ]:
def detect_mito_genes(adata):
    """
    Identify mitochondrial genes
    """

    print("\nDetecting mitochondrial genes...")

    adata.var["mt"] = adata.var_names.str.startswith("MT-")

    print(
        f"Number of mitochondrial genes: "
        f"{adata.var['mt'].sum()}"
    )

    return adata

##### 3.2 QC Metrics Calculation

* total counts
* number of detected genes
* mitochondrial percentage

In [ ]:
def calculate_qc_metrics(adata):
    """
    Calculate quality control metrics
    """

    print("\nCalculating QC metrics...")

    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt"],
        inplace=True
    )

    return adata

##### 3.3 QC Visualization Before Filtering

In [ ]:
def plot_qc_before_filtering(adata):
    """
    Visualize QC metrics before filtering
    """

    print("\nPlotting QC metrics before filtering...")

    sc.pl.violin(
        adata,
        [
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt"
        ],
        jitter=0.4,
        multi_panel=True
    )

##### 3.4 Spot Filtering

Remove low-quality spots.

In [ ]:
def filter_spots(adata):
    """
    Remove low-quality Visium spots
    """

    print("\nFiltering low-quality spots...")

    before = adata.n_obs

    sc.pp.filter_cells(
        adata,
        min_counts=500  # Can be adjusted later
    )

    after = adata.n_obs

    print(f"Spots before filtering: {before}")
    print(f"Spots after filtering : {after}")

    return adata

##### 3.5 Gene Filtering

Remove genes expressed in too fes spots to reduce noise.

In [ ]:
def filter_genes(adata):
    """
    Remove low-information genes
    """

    print("\nFiltering low-information genes...")

    before = adata.n_vars

    sc.pp.filter_genes(
        adata,
        min_cells=10
    )

    after = adata.n_vars

    print(f"Genes before filtering: {before}")
    print(f"Genes after filtering : {after}")

    return adata

##### 3.6 Normalization

Normalize counts per spot to remove sequencing depth differences.

In [ ]:
def normalize_data(adata):
    """
    Normalize total counts
    """

    print("\nNormalizing total counts...")

    sc.pp.normalize_total(
        adata,
        target_sum=1e4
    )

    return adata

##### 3.7 Log Transformation

Stabilize the highly skewed gene expression using: log(1 + x)

In [ ]:
def log_transform(adata):
    """
    Apply log1p transformation
    """

    print("\nApplying log transformation...")

    sc.pp.log1p(adata)

    return adata

##### 3.8 Highly Variable Gene (HVG) Selection

Keep only the most informative genes to improve clustering quality.

In [ ]:
def select_highly_variable_genes(adata, config):
    """
    Select highly variable genes
    """

    print("\nSelecting highly variable genes...")

    sc.pp.highly_variable_genes(
        adata,
        flavor="seurat",
        n_top_genes=config["n_top_genes"]
    )

    print(
        f"Number of HVGs selected: "
        f"{adata.var['highly_variable'].sum()}"
    )

    return adata

##### 3.9 HVG Visualization

In [ ]:
def plot_hvg(adata):
    """
    Visualize highly variable genes
    """

    print("\nPlotting highly variable genes...")

    sc.pl.highly_variable_genes(adata)

##### 3.10 Main Preprocessing Function

In [ ]:
def preprocess_data(adata, config):
    """
    Full preprocessing pipeline
    """

    adata = detect_mito_genes(adata)

    adata = calculate_qc_metrics(adata)

    plot_qc_before_filtering(adata)

    adata = filter_spots(adata)

    adata = filter_genes(adata)

    adata = normalize_data(adata)

    adata = log_transform(adata)

    adata = select_highly_variable_genes(
        adata,
        config
    )

    plot_hvg(adata)

    print("\nPreprocessing completed.")

    return adata

##### 3.11 Run

In [ ]:
adata = preprocess_data(adata, config)

### 4. Dimensionality Reduction and Clustering

##### 4.1 PCA

Reduce high-dimensional gene expression into informative components to help remove noise before clustering.

In [ ]:
def run_pca(adata):
    """
    Principal Component Analysis
    """

    print("\nRunning PCA...")

    sc.pp.pca(
        adata,
        use_highly_variable=True
    )

    return adata

##### 4.2 Neighborhood Graph Construction

Finds which spots are biologically similar; required for Leiden clustering.

In [ ]:
def build_neighbors(adata):
    """
    Build neighborhood graph
    """

    print("\nBuilding neighborhood graph...")

    sc.pp.neighbors(adata)

    return adata

##### 4.3 UMAP Projection

Creates a 2D visualization of biological similarity.

In [ ]:
def run_umap(adata):
    """
    UMAP projection
    """

    print("\nRunning UMAP...")

    sc.tl.umap(adata)

    return adata

##### 4.4 Leiden Clustering

Creates the actual unsupervised cluster IDs.

Example:
- cluster 0
- cluster 1 ...
- cluster 8

In [ ]:
def run_leiden_clustering(adata, config):
    """
    Leiden clustering
    """

    print("\nRunning Leiden clustering...")

    sc.tl.leiden(
        adata,
        resolution=config["leiden_resolution"]
    )

    print("\nClusters found:")
    print(
        adata.obs["leiden"].value_counts()
        .sort_index()
    )

    return adata

##### 4.5 Spatial Cluster Visualization

Shows where each cluster is located on tissue.

In [ ]:
def plot_spatial_clusters(adata):
    """
    Plot clusters on tissue image
    """

    print("\nPlotting spatial clusters...")

    sc.pl.spatial(
        adata,
        color="leiden",
        title="Spatial Leiden Clusters",
        alpha_img=0.5
    )

##### 4.6 UMAP Cluster Visualization

Shows cluster separation in latent space.

In [ ]:
def plot_umap_clusters(adata):
    """
    Plot UMAP clusters
    """

    print("\nPlotting UMAP clusters...")

    sc.pl.umap(
        adata,
        color="leiden",
        title="UMAP Leiden Clusters"
    )

##### 4.7 Save Clustered AnnData

In [ ]:
def save_clustered_adata(adata, paths):
    """
    Save clustered AnnData object
    """

    save_path = (
        paths["metadata_dir"] /
        "clustered_adata.h5ad"
    )

    adata.write(save_path)

    print(f"\nClustered AnnData saved to:\n{save_path}")

##### 4.8 Main Clustering Function

In [ ]:
def run_clustering(adata, config, paths):
    """
    Full clustering pipeline
    """

    adata = run_pca(adata)

    adata = build_neighbors(adata)

    adata = run_umap(adata)

    adata = run_leiden_clustering(
        adata,
        config
    )

    plot_spatial_clusters(adata)

    plot_umap_clusters(adata)

    save_clustered_adata(
        adata,
        paths
    )

    print("\nClustering completed.")

    return adata

##### 4.9 Run

In [ ]:
adata = run_clustering(adata, config, paths)

### 5. Marker Gene Analysis

Identifies the genes that define each Leiden cluster.

* Cluster 0 -> Stroma
* Cluster 1-7 -> Tumor
* Cluster 8 -> Immune

##### 5.1 Differential Expression Analysis

Find genes that are significantly enriched in each cluster to help identify what each cluster biologically represents.

In [ ]:
def run_marker_gene_analysis(adata):
    """
    Find marker genes for each Leiden cluster
    """

    print("\nRunning marker gene analysis...")

    sc.tl.rank_genes_groups(
        adata,
        groupby="leiden",
        method="t-test"
    )

    print("\nMarker gene analysis completed.")

    return adata

##### 5.2 View Top Marker Genes per Cluster

Prints the top marker genes for quick inspection.

In [ ]:
def print_top_marker_genes(adata, n_genes=10):
    """
    Print top marker genes per cluster
    """

    print("\nTop marker genes per cluster:\n")

    result = adata.uns["rank_genes_groups"]
    groups = result["names"].dtype.names

    for group in groups:
        genes = result["names"][group][:n_genes]

        print(f"Cluster {group}:")
        print(", ".join(genes))
        print("-" * 50)

##### 5.3 Dot Visualization

Shows which genes are highly expressed in which clusters.

In [ ]:
def plot_marker_dotplot(adata):
    """
    Dotplot for top marker genes
    """

    print("\nPlotting marker gene dotplot...")

    sc.pl.rank_genes_groups_dotplot(
        adata,
        n_genes=5,
        title="Top Marker Genes per Cluster",
        show=True
    )

##### 5.4 Heatmap Visualization

Helps visualize cluster-specific expression patterns.

In [ ]:
def plot_marker_heatmap(adata):
    """
    Heatmap for marker genes
    """

    print("\nPlotting marker gene heatmap...")

    sc.pl.rank_genes_groups_heatmap(
        adata,
        n_genes=5,
        show=True
    )

##### 5.5 Violin Plot Validation

Validate important known marker genes manually.

Examples:
* EPCAM -> Tumor
* CD3D -> Immune
* COL1A1 -> Stroma

In [ ]:
def plot_known_marker_validation(adata):
    """
    Validate known marker genes
    """

    marker_genes = [
        "EPCAM",
        "CD3D",
        "COL1A1"
    ]

    existing_markers = [
        gene for gene in marker_genes
        if gene in adata.var_names
    ]

    if len(existing_markers) == 0:
        print("\nNo known marker genes found.")
        return

    print("\nPlotting known marker validation...")

    sc.pl.violin(
        adata,
        keys=existing_markers,
        groupby="leiden"
    )

##### 5.6 Save Marker Results

In [ ]:
def save_marker_results(adata, paths):
    """
    Save marker gene results
    """

    result = sc.get.rank_genes_groups_df(
        adata,
        group=None
    )

    save_path = (
        paths["metadata_dir"] /
        "marker_genes.csv"
    )

    result.to_csv(save_path, index=False)

    print(f"\nMarker gene results saved to:\n{save_path}")

##### 5.7 Main Marker Analysis Function

In [ ]:
def run_marker_analysis(adata, paths):
    """
    Full marker gene analysis pipeline
    """

    adata = run_marker_gene_analysis(adata)

    print_top_marker_genes(adata)

    plot_marker_dotplot(adata)

    plot_marker_heatmap(adata)

    plot_known_marker_validation(adata)

    save_marker_results(
        adata,
        paths
    )

    print("\nMarker gene analysis completed.")

    return adata

##### 5.8 Run

In [ ]:
adata = run_marker_analysis(adata, paths)

### 6. Clinical Cluster Mapping

##### 6.1 Define Standard Marker Dictionary

Define a clinically meaningful marker reference to help justify why each cluster belongs to Tumor / Immune / Stroma.

In [ ]:
def get_marker_dictionary():
    """
    Standard marker dictionary for TME interpretation
    """

    marker_dict = {
        "Tumor": [
            "EPCAM", "KRT8", "KRT18", "KRT19",
            "EGFR", "CDK4"
        ],

        "Immune": [
            "CD3D", "CD3E", "CD3G",
            "CD8A", "CD8B",
            "NKG7", "GNLY",
            "CD79A", "MS4A1"
        ],

        "Stroma": [
            "COL1A1", "COL1A2",
            "DCN", "FAP",
            "ACTA2", "PDGFRB"
        ]
    }

    return marker_dict

##### 6.2 Define Manual Cluster Mapping

In [ ]:
def get_cluster_mapping():
    """
    Manual mapping from Leiden clusters
    to final clinical labels
    """

    cluster_mapping = {
        "0": "Stroma",

        "1": "Tumor",
        "2": "Tumor",
        "3": "Tumor",
        "4": "Tumor",
        "5": "Tumor",
        "6": "Tumor",
        "7": "Tumor",

        "8": "Immune"
    }

    return cluster_mapping

##### 6.3 Apply Clinical Labels

Convert leiden cluster ID into final TME label to create the actual supervision target.

In [ ]:
def apply_clinical_labels(adata):
    """
    Apply final clinical labels
    """

    print("\nApplying clinical labels...")

    cluster_mapping = get_cluster_mapping()

    adata.obs["tme_label"] = (
        adata.obs["leiden"]
        .map(cluster_mapping)
    )

    print("\nFinal label distribution:")
    print(
        adata.obs["tme_label"]
        .value_counts()
    )

    return adata

##### 6.4 Encodes Labels for Deep Learning

PyTorch needs integer labels, so we convert:
* Tumor: 0
* Immune: 1
* Stroma: 2

In [ ]:
def encode_labels(adata):
    """
    Convert string labels to integer labels
    """

    label_encoder = {
        "Tumor": 0,
        "Immune": 1,
        "Stroma": 2
    }

    adata.obs["label_id"] = (
        adata.obs["tme_label"]
        .map(label_encoder)
    )

    print("\nEncoded labels:")
    print(
        adata.obs[
            ["leiden", "tme_label", "label_id"]
        ].head()
    )

    return adata

##### 6.5 Spatial Visualization of Final Labels

This figure shows Tumor / Immune / Stroma directly on tissue

In [ ]:
def plot_final_labels(adata):
    """
    Visualize final TME labels on tissue
    """

    print("\nPlotting final clinical labels...")

    sc.pl.spatial(
        adata,
        color="tme_label",
        title="Final TME Labels",
        alpha_img=0.5
    )

##### 6.6 Save Final Label Table

Table to be used later for patch extraction and CNN training.

In [ ]:
def save_final_labels(adata, paths):
    """
    Save final labels table
    """

    save_df = adata.obs.copy()

    save_path = (
        paths["metadata_dir"] /
        "final_labels.csv"
    )

    save_df.to_csv(save_path)

    print(f"\nFinal labels saved to:\n{save_path}")

##### 6.7 Validation Summary Table

In [ ]:
def print_cluster_summary(adata):
    """
    Print cluster → label summary
    """

    summary = (
        adata.obs
        .groupby(["leiden", "tme_label"])
        .size()
        .reset_index(name="count")
    )

    print("\nCluster Mapping Summary:\n")
    print(summary)

##### 6.8 Main Clinical Mapping Function

In [ ]:
def run_clinical_mapping(adata, paths):
    """
    Full clinical mapping pipeline
    """

    marker_dict = get_marker_dictionary()

    print("\nMarker dictionary loaded:")
    print(marker_dict)

    adata = apply_clinical_labels(adata)

    adata = encode_labels(adata)

    plot_final_labels(adata)

    save_final_labels(
        adata,
        paths
    )

    print_cluster_summary(adata)

    print("\nClinical cluster mapping completed.")

    return adata

##### 6.9 Run

In [ ]:
adata = run_clinical_mapping(adata, paths)

### 7. Path Extraction from Histology Image

##### 7.1 Get Spatial Coordinates

Extract Visium spot coordinates that tell us where to crop image patches.

In [ ]:
def get_spatial_coordinates(adata, config):
    """
    Extract correctly scaled coordinates
    for hires image patch extraction
    """

    print("\nExtracting spatial coordinates...")

    library_id = config["sample_id"]

    coords = adata.obsm["spatial"].copy()

    scale = adata.uns["spatial"][library_id][
        "scalefactors"
    ]["tissue_hires_scalef"]

    coords = coords * scale

    print(f"Scale factor used: {scale}")
    print(f"Number of coordinates: {len(coords)}")

    return coords

##### 7.2 Load H&E Tissue Image

Load the high-resolution histology image (the image source for patch extraction).

In [ ]:
def load_histology_image(adata, config):
    """
    Load high-resolution tissue image
    """

    print("\nLoading histology image...")

    library_id = config["sample_id"]

    hires = (
        adata.uns["spatial"][library_id]
        ["images"]["hires"]
    )

    image_container = sq.im.ImageContainer(
        hires
    )

    print("Histology image loaded.")

    return image_container

##### 7.3 Patch Extraction Function

We crop 224 x 224 image patches centered at each Visium spot.

In [ ]:
def extract_patch(
    image_container,
    x,
    y,
    patch_size=224
):
    """
    Extract single image patch
    """

    half = patch_size // 2

    img = np.asarray(
        image_container["image"].values
    ).squeeze()

    h, w, _ = img.shape

    x = int(np.clip(x, half, w - half))
    y = int(np.clip(y, half, h - half))

    patch = img[
        y - half : y + half,
        x - half : x + half
    ]

    return patch

##### 7.4 Create Patch Metadata Table

Each row becomes:
* patch
* + coordinates
* + label

In [ ]:
def create_patch_metadata(
    adata,
    coords
):
    """
    Create metadata table for patch extraction
    """

    print("\nCreating patch metadata table...")

    df = pd.DataFrame({
        "barcode": adata.obs.index,   # IMPORTANT FIX
        "x": coords[:, 0],
        "y": coords[:, 1],
        "leiden": adata.obs["leiden"].values,
        "tme_label": adata.obs["tme_label"].values,
        "label_id": adata.obs["label_id"].values
    })

    print(df.head())

    return df

##### 7.5 Save Patch Images

Each patch gets:
* patch_0001.png
* patch_0002.png
* ...

In [ ]:
def save_patches(
    df,
    image_container,
    config,
    paths
):
    """
    Save all patches to disk
    """

    print("\nSaving image patches...")

    patch_dir = paths["patch_dir"]

    saved_paths = []

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):
        patch = extract_patch(
            image_container,
            x=row["x"],
            y=row["y"],
            patch_size=config["patch_size"]
        )

        save_path = (
            patch_dir /
            f"patch_{idx:04d}.png"
        )

        plt.imsave(save_path, patch)

        saved_paths.append(str(save_path))

    df["patch_path"] = saved_paths

    print("Patch saving completed.")

    return df

##### 7.6 Save Metadata CSV

This CSV becomes the input for PyTorch Dataset.

In [ ]:
def save_patch_metadata(df, paths):
    """
    Save patch metadata CSV
    """

    save_path = (
        paths["metadata_dir"] /
        "patch_metadata.csv"
    )

    df.to_csv(
        save_path,
        index=False
    )

    print(f"\nPatch metadata saved to:\n{save_path}")

##### 7.7 Quick Quality Check

Manual inspection of a few patches.

In [ ]:
def show_sample_patches(df, n=5):
    """
    Show sample extracted patches
    """

    print("\nShowing sample patches...")

    sample_df = df.sample(n)

    fig, axes = plt.subplots(
        1,
        n,
        figsize=(4 * n, 4)
    )

    for ax, (_, row) in zip(
        axes,
        sample_df.iterrows()
    ):
        img = plt.imread(
            row["patch_path"]
        )

        ax.imshow(img)
        ax.set_title(
            f"{row['tme_label']}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

##### 7.8 Main Patch Extraction Function

In [ ]:
def run_patch_extraction(
    adata,
    config,
    paths
):
    """
    Full patch extraction pipeline
    """

    coords = get_spatial_coordinates(
        adata,
        config
    )

    image_container = load_histology_image(
        adata,
        config
    )

    df = create_patch_metadata(
        adata,
        coords
    )

    df = save_patches(
        df,
        image_container,
        config,
        paths
    )

    save_patch_metadata(
        df,
        paths
    )

    show_sample_patches(df)

    print("\nPatch extraction completed.")

    return df

##### 7.9 Run

In [ ]:
df = run_patch_extraction(adata, config, paths)

### 8. PyTorch Dataset Construction

##### 8.1 Define Image Transform

Standardizes images before training.
Especially for ResNet, normalization is important.

In [ ]:
def get_image_transform(train=True):
    """
    Strong augmentation without black corner artifacts
    """

    if train:
        transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),

            transforms.RandomResizedCrop(
                size=224,
                scale=(0.85, 1.0),
                ratio=(0.95, 1.05)
            ),

            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.05
            ),

            transforms.ToTensor(),

            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    else:
        transform = transforms.Compose([
            transforms.ToTensor(),

            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    return transform

##### 8.2 Custom Dataset Class

Each sample returns:
* image tensor
* label_id
* metadata

In [ ]:
class HistologyPatchDataset(Dataset):
    """
    Patch classification dataset
    """

    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            row["patch_path"]
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            row["label_id"],
            dtype=torch.long
        )

        return {
            "image": image,
            "label": label,
            "tme_label": row["tme_label"],
            "leiden": row["leiden"],
            "x": row["x"],
            "y": row["y"]
        }

##### 8.3 Train / Validation Split

In [ ]:
def split_dataset(df):
    """
    Train / validation split
    """

    print("\nSplitting dataset...")

    train_df = df.sample(
        frac=0.8,
        random_state=42
    )

    val_df = df.drop(
        train_df.index
    )

    print(f"Train size: {len(train_df)}")
    print(f"Validation size: {len(val_df)}")

    return train_df, val_df

##### 8.4 Create Dataset Objects

In [ ]:
def create_datasets(train_df, val_df):
    """
    Create PyTorch Dataset objects
    """

    train_transform = get_image_transform(train=True)
    val_transform = get_image_transform(train=False)

    train_dataset = HistologyPatchDataset(
        train_df,
        transform=train_transform
    )

    val_dataset = HistologyPatchDataset(
        val_df,
        transform=val_transform
    )

    return train_dataset, val_dataset


def create_full_dataset(df):
    """
    Dataset for inference on ALL patches
    """

    transform = get_image_transform()

    full_dataset = HistologyPatchDataset(
        df,
        transform=transform
    )

    print(f"\nFull dataset size: {len(full_dataset)}")

    return full_dataset

##### 8.5 Create DataLoaders

DataLoader handles batching & shuffling.

In [ ]:
def create_dataloaders(
    train_dataset,
    val_dataset,
    config
):
    """
    Create DataLoaders
    """

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=2
    )

    return train_loader, val_loader

##### 8.6 Quick Batch Sanity Check

Expected:


```
Image shape:
[batch_size, 3, 224, 224]
```



In [ ]:
def check_batch(train_loader):
    """
    Quick batch check + visualization
    """

    print("\nChecking batch structure...")

    batch = next(iter(train_loader))

    print("Image shape:")
    print(batch["image"].shape)

    print("\nLabel shape:")
    print(batch["label"].shape)

    print("\nLabels:")
    print(batch["label"][:10])

    # Visualize sample images
    print("\nVisualizing sample batch images...")

    images = batch["image"][:6]   # show first 6
    labels = batch["label"][:6]

    # unnormalize for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    fig, axes = plt.subplots(
        2, 3,
        figsize=(12, 8)
    )

    axes = axes.flatten()

    label_decoder = {
        0: "Tumor",
        1: "Immune",
        2: "Stroma"
    }

    for i in range(6):
        img = images[i].cpu() * std + mean
        img = torch.clamp(img, 0, 1)

        img = img.permute(1, 2, 0).numpy()

        axes[i].imshow(img)
        axes[i].set_title(
            f"{label_decoder[int(labels[i])]}"
        )
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

##### 8.8 Main Dataset Construction Function

In [ ]:
def build_pytorch_dataset(
    df,
    config
):
    """
    Full PyTorch dataset pipeline
    """

    train_df, val_df = split_dataset(df)

    train_dataset, val_dataset = create_datasets(
        train_df,
        val_df
    )

    train_loader, val_loader = create_dataloaders(
        train_dataset,
        val_dataset,
        config
    )

    check_batch(train_loader)

    print("\nPyTorch dataset construction completed.")

    return (
        train_df,
        val_df,
        train_dataset,
        val_dataset,
        train_loader,
        val_loader
    )

##### 8.8 Run

In [ ]:
(train_df, val_df,
 train_dataset, val_dataset,
 train_loader, val_loader) = build_pytorch_dataset(df, config)

full_dataset = create_full_dataset(df)

### 9. CNN Model Definition

##### 9.1 ResNet18

As starting point.

##### 9.2 Image Encoder (Backbone)

Remove the original classification layer; only feature extraction remains.

In [ ]:
class ImageEncoder(nn.Module):
    """
    CNN backbone for histology feature extraction
    """

    def __init__(
        self,
        backbone="resnet18",
        pretrained=True
    ):
        super().__init__()

        if backbone == "resnet18":
            model = models.resnet18(
                weights="IMAGENET1K_V1"
                if pretrained else None
            )
            out_dim = 512

        elif backbone == "resnet34":
            model = models.resnet34(
                weights="IMAGENET1K_V1"
                if pretrained else None
            )
            out_dim = 512

        elif backbone == "resnet50":
            model = models.resnet50(
                weights="IMAGENET1K_V1"
                if pretrained else None
            )
            out_dim = 2048

        else:
            raise ValueError(
                f"Unsupported backbone: {backbone}"
            )

        model.fc = nn.Identity()

        self.encoder = model
        self.out_dim = out_dim

    def forward(self, x):
        return self.encoder(x)

##### 9.3 Classification Head

Predicts Tumor / Immune / Stroma for extracted image features.

In [ ]:
class ClassificationHead(nn.Module):
    """
    Final classification head
    """

    def __init__(
        self,
        in_dim,
        num_classes
    ):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.mlp(x)

##### 9.4 Full Model

encoder + classifier

In [ ]:
class TMEClassifier(nn.Module):
    """
    Histology → TME label classifier
    """

    def __init__(
        self,
        num_classes,
        backbone="resnet18",
        pretrained=True
    ):
        super().__init__()

        self.encoder = ImageEncoder(
            backbone=backbone,
            pretrained=pretrained
        )

        self.head = ClassificationHead(
            in_dim=self.encoder.out_dim,
            num_classes=num_classes
        )

    def forward(self, x):
        features = self.encoder(x)
        logits = self.head(features)

        return logits

##### 9.5 Model Builder

Initializes the model on GPU/CPU.

In [ ]:
def build_model(
    config,
    device
):
    """
    Initialize classification model
    """

    print("\nBuilding CNN model...")

    model = TMEClassifier(
        num_classes=config["num_classes"],
        backbone="resnet18",
        pretrained=True
    )

    model = model.to(device)

    print(model)

    return model

##### 9.6 Loss Function

Use CrossEntropyLoss because this is classification.

In [ ]:
def get_class_weights(train_df, device):
    """
    Compute class weights for imbalanced classes
    """

    class_counts = (
        train_df["label_id"]
        .value_counts()
        .sort_index()
    )

    print("\nClass counts:")
    print(class_counts)

    weights = 1.0 / class_counts.values
    weights = weights / weights.sum()

    class_weights = torch.tensor(
        weights,
        dtype=torch.float
    ).to(device)

    print("\nClass weights:")
    print(class_weights)

    return class_weights

In [ ]:
def get_loss_function(class_weights):
    return nn.CrossEntropyLoss(
        weight=class_weights
    )

##### 9.7 Optimizer

We use AdamW for starting point.

In [ ]:
def get_optimizer(
    model,
    config
):
    """
    Optimizer
    """

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    return optimizer

##### 9.8 Learning Rate Scheduler

In [ ]:
def get_scheduler(optimizer):
    """
    Learning rate scheduler
    """

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,
        T_mult=2
    )

    return scheduler

##### 9.9 Forward Pass Sanity Check

In [ ]:
def check_model_forward(
    model,
    train_loader,
    device
):
    """
    Quick forward pass check
    """

    print("\nChecking model forward pass...")

    batch = next(iter(train_loader))

    images = batch["image"].to(device)

    with torch.no_grad():
        outputs = model(images)

    print("Input shape:")
    print(images.shape)

    print("\nOutput shape:")
    print(outputs.shape)

    print("\nExpected output:")
    print(
        f"[batch_size, {3}]"
    )

##### 9.10 Main Model Setup Function

In [ ]:
def prepare_model(
    config,
    device,
    train_loader,
    train_df
):
    """
    Full model preparation
    """

    model = build_model(
        config,
        device
    )

    # class imbalance fix
    class_weights = get_class_weights(
        train_df,
        device
    )

    criterion = get_loss_function(
        class_weights
    )

    optimizer = get_optimizer(
        model,
        config
    )

    scheduler = get_scheduler(
        optimizer
    )

    check_model_forward(
        model,
        train_loader,
        device
    )

    print("\nModel preparation completed.")

    return (
        model,
        criterion,
        optimizer,
        scheduler
    )

##### 9.11 Run

In [ ]:
(model, criterion,
 optimizer, scheduler) = prepare_model(
    config, device, train_loader, train_df)

### 10. Model Training

##### 10.1 Training for Classification

* input = image patch
* target = label_id
* loss = CrossEntropyLoss

##### 10.2 Single Training Epoch

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):
    """
    Train for one epoch
    """

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in pbar:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += (
            loss.item() * images.size(0)
        )

        preds = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        acc = correct / total

        pbar.set_postfix(
            loss=loss.item(),
            acc=f"{acc:.4f}"
        )

    epoch_loss = total_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

##### 10.3 Validation Loop

Checks performance without updating weights.

In [ ]:
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    """
    Validation for one epoch
    """

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            total_loss += (
                loss.item() * images.size(0)
            )

            preds = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    epoch_loss = total_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

##### 10.4 Save Best Model

Save the "best" validation model.

In [ ]:
def save_best_model(
    model,
    val_acc,
    best_acc,
    paths
):
    """
    Save best checkpoint
    """

    if val_acc > best_acc:
        best_acc = val_acc

        save_path = (
            paths["checkpoint_dir"] /
            "best_model.pt"
        )

        torch.save(
            model.state_dict(),
            save_path
        )

        print(
            f"New best model saved "
            f"(Val Acc: {val_acc:.4f})"
        )

    return best_acc

##### 10.5 Training History Plot

For reports & debugging.

In [ ]:
def plot_training_history(
    train_losses,
    val_losses,
    train_accs,
    val_accs
):
    """
    Plot training curves
    """

    epochs = range(
        1,
        len(train_losses) + 1
    )

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(
        epochs,
        train_losses,
        label="Train Loss"
    )
    plt.plot(
        epochs,
        val_losses,
        label="Val Loss"
    )
    plt.legend()
    plt.title("Loss")

    plt.subplot(1, 2, 2)
    plt.plot(
        epochs,
        train_accs,
        label="Train Acc"
    )
    plt.plot(
        epochs,
        val_accs,
        label="Val Acc"
    )
    plt.legend()
    plt.title("Accuracy")

    plt.tight_layout()
    plt.show()

##### 10.6 Full Training Loop

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    config,
    device,
    paths
):
    """
    Full training pipeline + Early Stopping (Patience)
    """

    print("\nStarting model training...")

    best_acc = 0

    # Early stopping settings
    patience = config["patience"]
    patience_counter = 0

    train_losses = []
    val_losses = []

    train_accs = []
    val_accs = []

    for epoch in range(config["epochs"]):
        print(
            f"\nEpoch "
            f"{epoch+1}/{config['epochs']}"
        )

        # Train
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        # Validation
        val_loss, val_acc = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )

        scheduler.step()

        # Early Stopping Logic
        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0

            save_path = (
                paths["checkpoint_dir"] /
                "best_model.pt"
            )

            torch.save(
                model.state_dict(),
                save_path
            )

            print(
                f"New best model saved "
                f"(Val Acc: {val_acc:.4f})"
            )

        else:
            patience_counter += 1

            print(
                f"No improvement. "
                f"Patience: {patience_counter}/{patience}"
            )

        # Save history
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f}"
        )

        print(
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        # Stop if patience exceeded
        if patience_counter >= patience:
            print(
                f"\nEarly stopping triggered "
                f"after {epoch+1} epochs."
            )
            break

    # Final plots
    plot_training_history(
        train_losses,
        val_losses,
        train_accs,
        val_accs
    )

    print(
        f"\nTraining completed. "
        f"Best Val Acc: {best_acc:.4f}"
    )

##### 10.7 Run

In [ ]:
train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    config,
    device,
    paths
)

### 11. Inference and Spatial Mask Reconstruction

##### 11.1 Reconstruct

Until now, the model predicts one patch -> one label.

We want is whole tissue -> spatial organization.

This section reconstructs the full tissue landscape.

##### 11.2 Load Best Model

Use the saved best checkpoint.

In [ ]:
def load_best_model(
    model,
    paths,
    device
):
    """
    Load best trained model
    """

    save_path = (
        paths["checkpoint_dir"] /
        "best_model.pt"
    )

    model.load_state_dict(
        torch.load(
            save_path,
            map_location=device
        )
    )

    model.eval()

    print(
        f"\nBest model loaded from:\n{save_path}"
    )

    return model

##### 11.3 Predict All Patches

Run inference for every patch.

In [ ]:
def predict_all_patches(
    model,
    loader,
    device
):
    """
    Predict labels for all patches
    """

    print("\nRunning inference...")

    all_preds = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(loader):
            images = batch["image"].to(device)

            outputs = model(images)

            probs = torch.softmax(
                outputs,
                dim=1
            )

            preds = torch.argmax(
                probs,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_probs.extend(
                probs.cpu().numpy()
            )

    print("Inference completed.")

    return all_preds, all_probs

##### 11.4 Label Decoder

Convert integer predictions back to names.

In [ ]:
def decode_predictions(preds):
    """
    Decode label IDs
    """

    decoder = {
        0: "Tumor",
        1: "Immune",
        2: "Stroma"
    }

    decoded = [
        decoder[p]
        for p in preds
    ]

    return decoded

##### 11.5 Create Prediction Table

Stores all predictions for visualization.

In [ ]:
def create_prediction_table(dataset, preds):
    """
    Build prediction dataframe
    FIXED VERSION
    """

    print("\nCreating prediction table...")

    pred_labels = decode_predictions(preds)

    # IMPORTANT:
    # keep original dataframe with barcode preserved
    df_pred = dataset.df.copy().reset_index(drop=True)

    # sanity check
    if len(df_pred) != len(preds):
        raise ValueError(
            f"Length mismatch: df={len(df_pred)} vs preds={len(preds)}"
        )

    df_pred["pred_label"] = pred_labels
    df_pred["pred_id"] = preds

    print(df_pred.head())

    return df_pred

##### 11.6 Spatial Prediction Visualization

Overlays predicted labels onto tissue coordinates; Final prediction mask.

In [ ]:
def plot_prediction_map(adata, df_pred):
    """
    Plot predicted spatial labels
    FIXED VERSION
    """

    print("\nPlotting prediction mask...")

    # initialize column safely
    adata.obs["pred_label"] = pd.Series(
        index=adata.obs.index,
        dtype="object"
    )

    # VERY IMPORTANT:
    # use barcode column, not integer index
    valid_barcodes = df_pred["barcode"].isin(adata.obs.index)

    print(
        f"Matched barcodes: "
        f"{valid_barcodes.sum()} / {len(df_pred)}"
    )

    if valid_barcodes.sum() == 0:
        raise ValueError(
            "No matching barcodes found. "
            "Check barcode preservation."
        )

    matched_df = df_pred.loc[valid_barcodes].copy()

    # assign predictions correctly
    adata.obs.loc[
        matched_df["barcode"],
        "pred_label"
    ] = matched_df["pred_label"].values

    # final visualization
    sc.pl.spatial(
        adata,
        color="pred_label",
        title="Predicted TME Spatial Mask",
        alpha_img=0.5
    )

##### 11.7 Compare True vs Predicted Labels

In [ ]:
def compare_true_vs_pred(
    df_pred
):
    """
    Compare true labels vs predictions
    """

    comparison = pd.crosstab(
        df_pred["tme_label"],
        df_pred["pred_label"]
    )

    print("\nTrue vs Predicted Labels:\n")
    print(comparison)

In [ ]:
def plot_correct_vs_wrong(paths):
    """
    Visualize correct vs wrong predictions on spatial map
    """
    csv_path = (
        paths["metadata_dir"] /
        "prediction_results.csv"
    )

    df = pd.read_csv(csv_path)

    # Correct / incorrect flag
    df["is_correct"] = (
        df["tme_label"] == df["pred_label"]
    )

    correct = df[df["is_correct"]]
    wrong = df[~df["is_correct"]]

    plt.figure(figsize=(10, 10))

    # Correct predictions
    plt.scatter(
        correct["x"],
        correct["y"],
        s=12,
        alpha=0.5,
        label="Correct"
    )

    # Wrong predictions
    plt.scatter(
        wrong["x"],
        wrong["y"],
        s=30,
        alpha=0.9,
        label="Wrong"
    )

    # Histology coordinates usually need flipped y-axis
    plt.gca().invert_yaxis()

    plt.title(
        "Spatial Prediction Map\nCorrect vs Wrong Predictions"
    )

    plt.xlabel("X")
    plt.ylabel("Y")
    plt.legend()
    plt.tight_layout()
    plt.show()

##### 11.8 Save Final Prediction Results

In [ ]:
def save_prediction_results(
    df_pred,
    paths
):
    """
    Save prediction results
    """

    save_path = (
        paths["metadata_dir"] /
        "prediction_results.csv"
    )

    df_pred.to_csv(
        save_path,
        index=False
    )

    print(
        f"\nPrediction results saved to:\n{save_path}"
    )

##### 11.9 Full Inference Pipeline

In [ ]:
def run_inference_pipeline(
    model,
    full_loader,      # use FULL loader
    full_dataset,     # use FULL dataset
    adata,
    paths,
    device
):
    """
    Full inference + mask reconstruction
    FIXED VERSION
    """

    model = load_best_model(
        model,
        paths,
        device
    )

    preds, probs = predict_all_patches(
        model,
        full_loader,
        device
    )

    df_pred = create_prediction_table(
        full_dataset,
        preds
    )

    plot_prediction_map(
        adata,
        df_pred
    )

    compare_true_vs_pred(
        df_pred
    )

    save_prediction_results(
        df_pred,
        paths
    )

    plot_correct_vs_wrong(paths)

    print(
        "\nInference and spatial mask reconstruction completed."
    )

    return df_pred

##### 11.10 Run

In [ ]:
full_loader = DataLoader(
    full_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=2
)

df_pred = run_inference_pipeline(
    model,
    full_loader,
    full_dataset,
    adata,
    paths,
    device
)

We can see that the wrong labels are mostly from the borders of the three classes, which is expected due to ambiguity at such specific regions.

Yet, this inference is done on the full dataset (which has been already used for training and validation).

### 12. Clinical Validation and Interpretation

### 13. Final Figures

##### 13.1 Save Original Tissue Figure

In [ ]:
def save_original_tissue_figure(
    adata,
    paths
):
    """
    Figure 1
    """

    print("\nSaving Figure 1...")

    sc.pl.spatial(
        adata,
        color=None,
        title="Original Tissue + Visium Spots",
        show=False
    )

    save_path = (
        paths["figure_dir"] /
        "figure_1_original_tissue.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(save_path)

##### 13.2 Save Leiden Cluster Figure

In [ ]:
def save_leiden_cluster_figure(
    adata,
    paths
):
    """
    Figure 2
    """

    print("\nSaving Figure 2...")

    sc.pl.spatial(
        adata,
        color="leiden",
        title="Spatial Leiden Clusters",
        alpha_img=0.5,
        show=False
    )

    save_path = (
        paths["figure_dir"] /
        "figure_2_leiden_clusters.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(save_path)

##### 13.3 Save Marker Dotplot

In [ ]:
def save_marker_dotplot(
    adata,
    paths
):
    """
    Figure 3
    """

    print("\nSaving Figure 3...")

    sc.pl.rank_genes_groups_dotplot(
        adata,
        n_genes=5,
        title="Top Marker Genes per Cluster",
        show=False
    )

    save_path = (
        paths["figure_dir"] /
        "figure_3_marker_dotplot.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(save_path)

##### 13.4 Save Final Clinical Label Figure

In [ ]:
def save_final_label_figure(
    adata,
    paths
):
    """
    Figure 4
    """

    print("\nSaving Figure 4...")

    sc.pl.spatial(
        adata,
        color="tme_label",
        title="Final TME Labels",
        alpha_img=0.5,
        show=False
    )

    save_path = (
        paths["figure_dir"] /
        "figure_4_final_labels.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(save_path)

##### 13.5 Save Prediction Mask Figure

In [ ]:
def save_prediction_mask_figure(
    adata,
    paths
):
    """
    Figure 5
    """

    print("\nSaving Figure 5...")

    sc.pl.spatial(
        adata,
        color="pred_label",
        title="Predicted TME Spatial Mask",
        alpha_img=0.5,
        show=False
    )

    save_path = (
        paths["figure_dir"] /
        "figure_5_prediction_mask.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(save_path)

##### 13.6 Save Confusion Matrix Figure

In [ ]:
def save_confusion_matrix_figure(
    df_pred,
    paths
):
    """
    Figure 6
    """

    print("\nSaving Figure 6...")

    cm = confusion_matrix(
        df_pred["tme_label"],
        df_pred["pred_label"],
        labels=["Tumor", "Immune", "Stroma"]
    )

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["Tumor", "Immune", "Stroma"],
        yticklabels=["Tumor", "Immune", "Stroma"]
    )

    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")

    save_path = (
        paths["figure_dir"] /
        "figure_6_confusion_matrix.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(save_path)

##### 13.7 Final Figure Summary

In [ ]:
def print_final_figure_summary(paths):
    """
    Final summary
    """

    print("\nFinal Figures Saved:\n")

    for file in sorted(
        paths["figure_dir"].glob("*.png")
    ):
        print(file)

    print("""
Recommended presentation order:

1. Original tissue
2. Leiden clusters
3. Marker dotplot
4. Clinical labels
5. Prediction mask
6. Confusion matrix
7. Training curves
""")

##### 13.8 Full Figure Generation Pipeline

In [ ]:
def generate_final_figures(
    adata,
    df_pred,
    paths
):
    """
    Full final figure pipeline
    """

    save_original_tissue_figure(
        adata,
        paths
    )

    save_leiden_cluster_figure(
        adata,
        paths
    )

    save_marker_dotplot(
        adata,
        paths
    )

    save_final_label_figure(
        adata,
        paths
    )

    save_prediction_mask_figure(
        adata,
        paths
    )

    save_confusion_matrix_figure(
        df_pred,
        paths
    )

    print_final_figure_summary(
        paths
    )

    print(
        "\nFinal figure generation completed."
    )

##### 13.9 Run

In [ ]:
generate_final_figures(
    adata,
    df_pred,
    paths
)

### 14. Inference on Unseen Data